# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR^2 dataset using the `mlcroissant` library. All dataset elements—record sets, fields, and columns—are referenced by their unique `@id` identifiers, in line with Croissant best practices.

### Dataset Source
This dataset is sourced from a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is available
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`. The Croissant schema uniquely identifies all data elements using `@id` fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's explore the available record sets and associated field `@id`s in the dataset. All references use the Croissant entity `@id` for precision and reproducibility.

In [ ]:
# List available record sets with their @id and fields
record_sets = list(dataset.record_sets)
print("Available Record Sets (@id, name):")
for rs in record_sets:
    print(f"- @id: {rs['@id']}\n  name: {rs.get('name', '<no name>')}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for f in fields:
            if isinstance(f, dict):
                print(f"    Field @id: {f['@id']} (name: {f.get('name', '<no name>')})")
            else:
                print(f"    Field @id: {f}")
    else:
        print("    <No fields found>")

## 3. Data Extraction

We load data from each record set into pandas DataFrames for further analysis. All references use record set and field `@id`s from the previous overview.

In [ ]:
# For demonstration, extract all record sets. Use @id when referencing.
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Each record from this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set @id: {record_set_id}")
    else:
        print(f"No records found for record set @id: {record_set_id}")

if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nFirst loaded DataFrame columns for record set @id: {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()
else:
    print("No tabular record sets with records found.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filter records, normalize numeric fields, and group by categorical fields. Use `@id` variables to reference columns, ensuring consistency with Croissant's approach.

In [ ]:
# For demonstration, use the first loaded tabular record set
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}")
    print("All column @ids:", df.columns.tolist())

    # Attempt to automatically pick a numeric field (int or float type)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field = numeric_fields[0]  # Use the first numeric column
        threshold = 10

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a likely categorical field (one with few unique values, but not the numeric)
        categorical_fields = [col for col in df.columns if df[col].dtype == object and df[col].nunique() < 20]
        group_field = None
        for col in categorical_fields:
            if col != numeric_field:
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical (group) field found.")
    else:
        print("No numeric fields found in this record set.")
else:
    print("No dataframes to analyze.")

## 5. Visualization

Visualize distributions or field relationships using matplotlib and seaborn. Again, reference fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals():
    # Histogram of the chosen numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if 'group_field' in locals() and group_field:
        # Boxplot by group
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- In this notebook, we have loaded and explored the FAIR^2 clinical oncology dataset using `mlcroissant`, referencing all entities by their `@id` as per the Croissant standard.
- We demonstrated how to list record sets and fields, extract records to pandas DataFrames, and carry out basic EDA and visualization.
- For further analysis, you can extend this notebook by referencing additional fields and record sets by their `@id`, and by applying domain-specific analytical methods.